# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [ ]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())

---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning


### 1.1 — Dataset Overview
First look at the structure: columns, types, and a few sample rows.

In [ ]:
# Working copy — we keep df_raw untouched for later (Part 4 reveal)
df = df_raw.copy()

print("=== Shape ===")
print(df.shape)

print("\n=== Column names & dtypes ===")
print(df.dtypes)

print("\n=== First 5 rows ===")
display(df.head())

print("\n=== Basic statistics ===")
display(df.describe(include='all'))

### 1.2 — Missing Values & Duplicates

In [ ]:
# --- Missing values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ No missing values detected.")
else:
    print("⚠️ Missing values found:")
    display(missing_df)

# --- Duplicates ---
n_duplicates = df.duplicated().sum()
print(f"\nDuplicated rows: {n_duplicates}")
if n_duplicates > 0:
    df = df.drop_duplicates()
    print(f"  → Dropped {n_duplicates} duplicate rows. New shape: {df.shape}")

### 1.3 — Class Imbalance: Machine Failure Rate

> **Key constraint:** We drop `Machine failure` before training — it is only used at the very end (Part 4) to evaluate our unsupervised models. Here we just peek at the imbalance to calibrate the `contamination` hyperparameter in Part 2.

In [ ]:
failure_rate = df['Machine failure'].value_counts(normalize=True) * 100
print("Machine failure distribution:")
print(failure_rate.rename({0: 'Normal', 1: 'Failure'}).round(2))

fig, ax = plt.subplots(figsize=(5, 3))
df['Machine failure'].value_counts().rename({0: 'Normal', 1: 'Failure'}).plot(
    kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='black'
)
ax.set_title('Class Distribution: Machine Failure')
ax.set_ylabel('Count')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

true_contamination = df['Machine failure'].mean()
print(f"\n→ True contamination rate: {true_contamination:.4f} ({true_contamination*100:.2f}%)")
print("  We will use this as a reference to set the `contamination` hyperparameter in Part 2.")

### 1.4 — Distribution of Numerical Features

In [ ]:
numerical_cols = ['Air temperature [K]', 'Process temperature [K]',
                  'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    # Histogram + KDE
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue', bins=40, alpha=0.7)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.2, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {df[col].median():.1f}')
    ax.set_title(col)
    ax.legend(fontsize=8)

# Skewness summary in the last subplot
ax = axes[5]
skewness = df[numerical_cols].skew().sort_values()
colors = ['tomato' if abs(s) > 1 else 'steelblue' for s in skewness]
skewness.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Skewness per feature\n(|skew| > 1 = red)')
ax.set_xlabel('Skewness')

plt.suptitle('Distribution of Numerical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nSkewness values:")
print(df[numerical_cols].skew().round(3))

### 1.5 — Outlier Detection via Boxplots

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, col in enumerate(numerical_cols):
    ax = axes[i]
    # Color points by failure label for visual context
    normal = df[df['Machine failure'] == 0][col]
    failure = df[df['Machine failure'] == 1][col]
    ax.boxplot([normal, failure], labels=['Normal', 'Failure'],
               patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Boxplots: Normal vs Failure observations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# IQR-based outlier count per feature
print("\nIQR-based outlier counts:")
for col in numerical_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"  {col:35s}: {n_out:4d} outliers ({n_out/len(df)*100:.2f}%)")

### 1.6 — Correlation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pearson correlation heatmap
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=axes[0], linewidths=0.5)
axes[0].set_title('Pearson Correlation Matrix\n(numerical features)', fontweight='bold')

# Correlation with Machine failure
corr_with_target = df[numerical_cols + ['Machine failure']].corr()['Machine failure'].drop('Machine failure').sort_values()
colors = ['tomato' if c > 0 else 'steelblue' for c in corr_with_target]
corr_with_target.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Correlation with Machine failure\n(for EDA context only)', fontweight='bold')
axes[1].set_xlabel('Pearson r')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Air temp & Process temp are highly correlated (thermal inertia)")
print("  - Torque is positively correlated with failure")
print("  - Rotational speed is negatively correlated with failure")

### 1.7 — Categorical Feature: `Type`

The `Type` column represents the machine quality tier (L=Low, M=Medium, H=High).  
Distance-based algorithms (LOF, EllipticEnvelope, OneClassSVM) cannot handle strings — we need to encode it numerically.

**Strategy chosen: Ordinal encoding** (L=0, M=1, H=2), since the categories have a natural quality order.  
One-Hot Encoding would also be valid but adds dimensionality unnecessarily for only 3 levels.

In [ ]:
# Distribution of Type
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

type_counts = df['Type'].value_counts()
type_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'orange', 'green'],
                 edgecolor='black', rot=0)
axes[0].set_title('Machine Type Distribution')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')

# Failure rate per Type
failure_by_type = df.groupby('Type')['Machine failure'].mean() * 100
failure_by_type.plot(kind='bar', ax=axes[1], color=['steelblue', 'orange', 'green'],
                     edgecolor='black', rot=0)
axes[1].set_title('Failure Rate by Machine Type (%)')
axes[1].set_ylabel('Failure rate (%)')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom')

plt.suptitle('Type Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Ordinal encoding
type_mapping = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_mapping)
print(f"Type encoded: {type_mapping}")
print(df[['Type', 'Type_encoded']].value_counts().sort_index())

### 1.8 — Preprocessing: Drop Labels & Standardize Features

**Why standardize?**
- **LOF** and **EllipticEnvelope** compute Euclidean distances — features with large scales (e.g., RPM ~1500 vs Torque ~40) would dominate the distance calculation and bias the results.
- **OneClassSVM** uses kernel functions sensitive to feature magnitude.
- **IsolationForest** splits features recursively and is theoretically scale-invariant, but standardizing still improves consistency and comparison fairness.

**Constraint respected:** `Machine failure` is saved as `y_true` and dropped from the training set.

In [ ]:
from sklearn.preprocessing import StandardScaler

# --- Save ground truth labels (used ONLY in Part 4) ---
y_true = df['Machine failure'].values
print(f"Ground truth saved: {y_true.sum()} failures out of {len(y_true)} observations")

# --- Select features for modeling ---
# Drop: UDI (ID), Product ID (string ID), Type (replaced by encoded),
#        Machine failure (target), failure sub-types (TWF, HDF, PWF, OSF, RNF)
cols_to_drop = ['UDI', 'Product ID', 'Type', 'Machine failure',
                'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Keep only columns that exist in the dataframe
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

feature_cols = [c for c in df.columns if c not in cols_to_drop]
print(f"\nFeatures used for modeling ({len(feature_cols)}): {feature_cols}")

X_raw = df[feature_cols].copy()

# --- Standardization ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print("\n=== Before scaling — means and stds ===")
print(X_raw.describe().loc[['mean', 'std']].round(2))

print("\n=== After scaling — means ≈ 0, stds ≈ 1 ===")
print(X_scaled.describe().loc[['mean', 'std']].round(4))

### 1.9 — Pairplot: Visualizing Feature Interactions

A pairplot on the scaled features, coloured by machine failure status, gives intuition on which feature pairs best separate normal from anomalous behaviour.

In [ ]:
# Pairplot on scaled data (subsample for speed)
plot_df = X_scaled.copy()
plot_df['Failure'] = y_true

# Subsample to keep rendering fast
sample_idx = np.random.choice(len(plot_df), size=min(2000, len(plot_df)), replace=False)
plot_df_sample = plot_df.iloc[sample_idx]

g = sns.pairplot(plot_df_sample, hue='Failure', palette={0: 'steelblue', 1: 'tomato'},
                 diag_kind='kde', plot_kws={'alpha': 0.3, 's': 15},
                 vars=feature_cols)
g.fig.suptitle('Pairplot of Scaled Features (blue=Normal, red=Failure)',
               y=1.02, fontsize=13, fontweight='bold')
plt.show()

print("\nObservations:")
print("  - Failures (red) cluster in specific regions of Torque vs Rotational Speed")
print("  - Tool Wear shows a clear right-tail for failures")
print("  - Temperature features are tightly correlated — minimal extra info from both")

### 1.10 — EDA Summary & Preprocessing Decisions

| Step | Finding | Decision |
|------|---------|----------|
| Missing values | None detected | No imputation needed |
| Duplicates | None detected | No rows dropped |
| Class imbalance | ~3.4% failures | `contamination ≈ 0.034` in Part 2 |
| Skewness | `Rotational speed` moderately right-skewed | Acceptable after standardization |
| Outliers (IQR) | Present in Torque & Rotational speed | Expected — these are the true anomalies |
| Correlation | Air temp ↔ Process temp: r ≈ 0.88 | Both kept (they carry distinct signals) |
| `Type` encoding | 3 ordinal categories (L/M/H) | Ordinal encoding: L=0, M=1, H=2 |
| Scaling | Required for LOF, EllipticEnvelope, OneClassSVM | StandardScaler applied to all features |
| Labels | `Machine failure` dropped before training | Saved as `y_true` for Part 4 evaluation |

**Final feature matrix `X_scaled`:** shape `(10 000, 6)` — ready for modeling.

---
## Part 2: Modeling & Hyperparameter Tuning
*(Train IsolationForest, OneClassSVM, LocalOutlierFactor, and EllipticEnvelope. Remember to tune your threshold parameters!)*


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

# Your code here


---
## Part 3: Technical Comparison & Visualizations
*(Use PCA/t-SNE to visualize the flagged anomalies. Find an anomaly caught by one model but missed by another and explain why.)*


In [ ]:
from sklearn.decomposition import PCA

# Your code here


---
## Part 4: Managerial Conclusion & Business Strategy
*(Bring back `y_true`. Calculate the number of False Positives and False Negatives for each model. Apply the cost matrix. Which model wins?)*


In [ ]:
# Business Cost Matrix
COST_FP = 500     # False Positive: Wasted technician check
COST_FN = 15000   # False Negative: Catastrophic machine breakdown

# Your code here
